<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: فاديم فوسكريسينسكي، فاديمفوسكريسينسكي
    
## <center> توقع المشاريع الناجحة على Kickstarter



### الجزء الأول. وصف مجموعة البيانات والميزات



قبل الغوص في تحليل البيانات وهندسة الميزات، اسمحوا لي أن أقدم لكم سياق المشروع الحالي. في المشروع، أعمل مع البيانات المأخوذة من [منصة Kickstarter] عبر الإنترنت (https://www.kaggle.com/kemical/kickstarter-projects). تم إنشاء هذه المنصة خصيصًا للتسلية الجماعية، ويمكن لأي شخص (أو مجموعة من الأشخاص) نشر فكرة أي مشروع عليها ويتوقع أن يقوم بعض المستخدمين المهتمين بالمنصة بتمويل هذه الفكرة. عادة، يحصل الممولون على بعض الجوائز الحصرية من المبدعين، ويعتمد حجم الجائزة على مقدار الأموال التي تضعها في المشروع. من الواضح أنه ليس كل المشاريع تنجح وتحصل على ما يكفي من المال. ولذلك، فإن التنبؤ بتلك الميزات التي تجعل المشاريع ناجحة يصبح غرضًا مهمًا جدًا لمحللي البيانات. يمكن أن تكون نماذج العمل الجيد أدوات مفيدة جدًا لهؤلاء المبتكرين الذين يرغبون في تجربة حظهم في الحصول على المال من خلال المنصة.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from scipy import stats
import math
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support


لذا، فلنقم الآن بتحميل البيانات وإلقاء نظرة على محتوياتها. 


In [ ]:
df = pd.read_csv('ks-projects-201801.csv')


في مجموعة البيانات لدينا، لدينا 15 متغيرًا و378661 ملاحظة.


In [ ]:
df.shape


اسمحوا لي أن أصف المتغيرات في مجموعة البيانات.
*المعرف* - ليس متغيرًا مهمًا بالنسبة لنا والذي يتوافق فقط مع الرقم الفريد للمشروع.
*الاسم* - اسم المشروع
*الفئة* - الفئة الفرعية
*فئة_رئيسية* - فئة واسعة يرتبط بها المشروع الحالي
*العملة* - العملة التي يجب تنفيذ الدفعات بها
*الموعد النهائي* - التاريخ الذي لن يتم بعده قبول الدفعات
*الهدف* - الأموال اللازمة لتحقيق المشروع*launched* - عندما تم إطلاق المشروع
*التعهد* - مقدار التمويل الذي قام به الأشخاص
*الحالة (المتغير المستهدف)* - ما الذي يحدث أخيرًا بالمشروع بعد الموعد النهائي (سواء كان فاشلًا أو ملغيًا أو ناجحًا)
*الداعمين* - عدد الأشخاص الذين قاموا بتمويل المشروع
*البلد* - بلد منشئي المشروع
*usd_pledged* - المبلغ الذي يدفعه الداعمون (بالدولار الأمريكي)
*usd_pledged_real* - المبلغ الذي يدفعه الداعمون (يتم تحويله إلى الدولار الأمريكي بواسطة تطبيق آخر)
*usd_goal* - الهدف (يتم تحويله إلى الدولار الأمريكي)


In [ ]:
df.head(5)


### الجزء 2-3. تحليل البيانات الاستكشافية والمرئية



أولا وقبل كل شيء، قبل النظر في التبعيات بين المتغيرات لا بد لي من النظر في توزيعات الميزات والمتغير المستهدف.
لنبدأ بالمتغير المستهدف. ما هي القيم التي تشملها؟


In [ ]:
df['state'].unique()


لذا، يمكننا أن نرى أنه بصرف النظر عن المشاريع الفاشلة والناجحة، فقد قمنا أيضًا بإلغاء المشاريع الحية وغير المحددة والمعلقة. دعونا ننظر كيف يتم توزيع مشاريعنا بين هذه الفئات.


In [ ]:
df.groupby('state').size().plot(kind='bar', color = "blue")


ترتبط معظم البيانات بالفئات "الفاشلة" و"الناجحة". كما يمكننا أن نرى أن بياناتنا غير متوازنة بعض الشيء؛ لدينا مشاريع فاشلة أكثر من المشاريع الناجحة.
دعونا نترك فقط المشاريع الفاشلة والناجحة. 


In [ ]:
df = df[df['state'].isin(['failed', 'successful'])]
df.shape


الآن دعنا ننتقل إلى ميزاتنا.
أولاً، من المثير للاهتمام التحقق من الفئات الأكثر شعبية على Kickstarter.


In [ ]:
df.groupby('main_category').size().sort_values(ascending = False).plot(kind='bar', color = "blue")


يمكننا أن نرى أن المشاريع المتعلقة بفئات الأفلام والفيديو تشكل الجزء الأكبر من جميع المشاريع في مجموعة البيانات الخاصة بنا. 
بمساعدة العمود الذي يمثل الفئات الفرعية، يمكننا معرفة أنواع المشاريع المدرجة في هذه الفئة.


In [ ]:
df[df.main_category == 'Film & Video'].groupby('category').size().sort_values(ascending = False).plot(kind='bar', color = "blue")

حسنًا، يبدو أن معظم المشاريع في فئة الأفلام والفيديو تحتاج إلى بعض المال لرعاية الأفلام الوثائقية والقصيرة. وفي الوقت نفسه، نلاحظ أن ما يقرب من 8000 فيلم لا ترتبط بأي فئة فرعية ويتم تصنيفها على أنها أفلام وفيديو. 
ماذا يحدث مع الدول. ومن أي الدول تأتي المشاريع؟


In [ ]:
df.groupby('country').size().head(20).sort_values(ascending = False).plot(kind='bar', color = "blue")


إذا نظرنا إلى الدول العشرين الأكثر شعبية، فسنرى أن بريطانيا العظمى هي الرائدة بلا منازع، ومعظم المشاريع تأتي من هذا البلد بالضبط.
الآن، دعونا ننظر كيف يتم توزيع المتغيرات العددية لدينا. لنبدأ بعدد المؤيدين. بالنسبة للتصور، أخذت اللوغاريتم لعدد الداعمين حيث أن لدينا قيم متطرفة قوية جدًا في هذا المتغير، ويتم ضغط جميع القيم الصغيرة في أسفل المخطط. 


In [ ]:
sns.boxplot(np.log(np.array(df['backers']) +1))


وفقًا لمخطط boxplot، يمكننا أن نرى أن لدينا بعض المشاريع النادرة التي لديها أكثر من 200000 داعم، في حين أن متوسط التوزيع الحقيقي (غير اللوغاريتمي) هو 15. 
كيف تبدو المخططات الصندوقية للأهداف والأموال المتعهد بها؟ بالنسبة لهذه الميزات، أخذت المتغيرات التي تم تحويلها بالفعل إلى الدولار الأمريكي.


In [ ]:
sns.boxplot(np.log(np.array(df['usd_goal_real']) +1))


توزيع الهدف المسجل قريب جدًا من الهدف الطبيعي حيث تبلغ القيمة المتوسطة الحقيقية 5000 دولار أمريكي. وفي هذه الحالة، لدينا قيم متطرفة من كلا جانبي مخطط الصندوق.



إذن، نحن الآن نعرف بعض المعلومات عن متغيراتنا المستقلة والتابعة، ويمكننا الانتقال إلى الجزء الأكثر إثارة للاهتمام من EDA حيث سنقوم بدراسة العلاقات بين هذه المتغيرات.
أولاً، دعونا ندرس الفئات التي حققت نجاحًا أكبر من حيث التمويل.قبل تصور الأنماط، سأقوم بإحصاء اختبار Chi2 للتوافق للتحقق مما إذا كان هناك أي اتصال مهم إحصائيًا بين المتغيرات المرصودة. في حالة اختبار chi2، تقول الفرضية الصفرية أن المتغيرات الفئوية مستقلة عن بعضها البعض، في حين تدعي الفرضية البديلة أن هذه المتغيرات مترابطة. 


In [ ]:
def chi2_output(x, y):
    results = stats.chi2_contingency(pd.crosstab(x, y))
    return "X2: {0}, p-value: {1}, degrees of freedom: {2}".format(results[0], results[1], results[2])

In [ ]:
chi2_output(df.state, df.main_category)


إذن، القيمة p = 0، وهذا يعني أنه يمكننا رفض الفرضية الصفرية والقول بأن متغيراتنا مترابطة.
للنظر بالتفصيل في كيفية ترابط المتغيرات، سأكتب دالة تحسب Pearson residlas (الفرق الموحد بين القيم المرصودة والمتوقعة) وترسم خريطة حرارية على أساسها. 


In [ ]:
def pearson_heatmap(x, y):
    crosst = pd.crosstab(x, y)
    chi2_stat, p_val, dof, ex = stats.chi2_contingency(crosst)
    pears_failed = (crosst.iloc[0,:] - ex[0])/np.sqrt(ex[0])
    pears_success = (crosst.iloc[1,:] - ex[1])/np.sqrt(ex[1])
    return sns.heatmap(pd.concat([pears_failed, pears_success], axis=1), annot=True,
                fmt=".1f",
                annot_kws={'size':10})

In [ ]:
a4_dims = (11.7, 8.27)
fig, ax = pyplot.subplots(figsize=a4_dims)
pearson_heatmap(df.state, df.main_category)


يمكننا أن نرى بعض الأنماط المثيرة للاهتمام من هذه الخريطة الحرارية. تتمتع المشاريع المتعلقة بفئات المسرح والموسيقى والقصص المصورة بفرص كبيرة للحصول على التمويل الكافي. في الوقت نفسه، غالبًا ما تفشل المشاريع المتعلقة بالطعام والأزياء والتكنولوجيا بشكل خاص. ربما يمكن تفسير ذلك بأن هذه المشاريع تحتاج إلى المزيد من المال. دعونا نتحقق من ذلك.


In [ ]:
a4_dims = (11.7, 8.27)
fig, ax = pyplot.subplots(figsize=a4_dims)
df_comp = df[df.main_category.isin(['Technology', 'Food', 'Fashion', 'Music', 'Comics', 'Theater'])]
sns.boxplot(df_comp["main_category"], np.log(df_comp['usd_goal_real']))


يمكننا بالتأكيد أن نرى أن متوسط قيمة الهدف للتكنولوجيا أعلى من الفئات الأخرى.
ماذا لو نظرنا إلى الفئات الفرعية للفئتين الأكثر شيوعًا في مجموعة البيانات لدينا (الأفلام والفيديو والموسيقى).


In [ ]:
chi2_output(df.state, df.category[df.main_category == "Film & Video"])


حسنًا، العلاقة ذات أهمية إحصائية.


In [ ]:
a4_dims = (11.7, 8.27)
fig, ax = pyplot.subplots(figsize=a4_dims)
pearson_heatmap(df.state, df.category[df.main_category == "Film & Video"])


آها! إذا كنت تصنع فيلمًا قصيرًا، فمن المحتمل أن تحصل على ما يكفي من المال.
ماذا عن الموسيقى؟


In [ ]:
chi2_output(df.state, df.category[df.main_category == "Music"])

In [ ]:
a4_dims = (11.7, 8.27)
fig, ax = pyplot.subplots(figsize=a4_dims)
pearson_heatmap(df.state, df.category[df.main_category == "Music"])


لا تطلب مساعدة مالية لإنشاء ألبوم هيب هوب، فمن الأسهل الحصول على تمويل كافٍ إذا كنت موسيقيًا مستقلًا أو كلاسيكيًا أو ريفيًا.
دعونا ننظر كيف يتغير نجاح/فشل المشروع اعتمادًا على البلد الأصلي.


In [ ]:
chi2_output(df.state, df.country)


المتغيرات مترابطة.


In [ ]:
a4_dims = (11.7, 8.27)
fig, ax = pyplot.subplots(figsize=a4_dims)
pearson_heatmap(df.state, df.country)

أنجح المشاريع تأتي من الولايات المتحدة، في حين أن المشاريع الإيطالية في كثير من الأحيان لا تحصل على التمويل الكافي.
دعونا نرى كيف يتفاعل المتغير المستهدف مع المتغيرات الرقمية المستقلة. نبدأ بالهدف.


In [ ]:
sns.boxplot(df["state"], np.log(df['usd_goal_real']))


كما هو متوقع، فإن المشاريع الفاشلة لها هدف متوسط أعلى.
ماذا عن الداعمين؟


In [ ]:
sns.boxplot(df["state"], np.log(df['backers'] + 1))


آها، ينبغي أن تكون ميزة جيدة للنموذج المستقبلي. المشاريع الناجحة لديها المزيد من الداعمين.



### الجزء الرابع. الأنماط والرؤى وخصائص البيانات 



كما يمكننا أن نرى من تحليل البيانات الاستكشافية أن هناك بعض العلاقات المثيرة للاهتمام بين ميزاتنا المحتملة والمتغير المستهدف (نجاح المشروع).
1. من المفترض أن إحدى أهم الميزات في نموذجنا المستقبلي ستكون الفئة التي يرتبط بها المشروع. ونظرًا للاختبارات الإحصائية المطبقة والخرائط الحرارية المرسومة، يمكننا أن نرى أنه ليست كل الفئات لديها فرص متساوية للحصول على الدعم المالي الكافي. 
2. علاوة على ذلك، وجدنا تمايزًا قويًا داخل الفئات. على سبيل المثال، الإستراتيجية الأفضل لجمع ما يكفي من المال في فئة الأفلام والفيديو هي إنتاج أفلام قصيرة، وإذا كنت تصنع موسيقى الهيب هوب، فلديك فرص أقل لجمع ما يكفي من الموارد مقارنة بالأشخاص الذين يصنعون موسيقى الروك المستقلة أو الموسيقى الكلاسيكية أو الشعبية. 
3. ولا نحتاج أيضًا إلى نسيان البلد الأصلي للمشروع. سواء كان المشروع يأتي من إيطاليا أو الولايات المتحدة يمكن أن يكون أحد العوامل الرئيسية التي تفسر نجاح المشروع. أعتقد أن الأمر مرتبط بشكل مباشر بشعبية المنصة في البلدان. 
4. وأيضا، بعد التحليل الاستكشافي، لدينا بعض المعلومات حول الخصائص المالية للمشاريع الناجحة. وفقا لتحليلنا، فإن المشاريع الناجحة لديها عدد أكبر من الداعمين وأهداف أقل. إنها نتيجة واضحة، لكنها، على أية حال، يمكن أن تكون مؤشرًا مفيدًا في نموذجنا.


### الجزء الخامس. المعالجة المسبقة للبيانات



أولاً، نحتاج إلى تحويل متغيراتنا الفئوية التي سنستخدمها (الفئة، الفئة_الرئيسية، الدولة) في النموذج النهائي إلى متغيرات وهمية. 


In [ ]:
dummies = pd.get_dummies(df.iloc[:, [2,3,11]])
dummies.shape


الآن، لدينا 197 عمودًا بجميع الفئات والفئات الفرعية.
ماذا عن المتغيرات الرقمية؟ على أساس تحليلنا الاستكشافي، يمكننا أن نستنتج أن المتغيرات # من الداعمين والهدف بالدولار الأمريكي يمكن أن تكون ميزات مهمة جدًا في نموذجنا. دعونا نضيفها إلى المتغيرات الوهمية.
ونحن بحاجة إلى حالة المتغير المستهدف لدينا والتي سيتم تحويلها إلى تنسيق ثنائي حيث يرمز 0 إلى الفشل و1 إلى النجاح.


In [ ]:
y = df['state'].map({'successful': 1, 'failed': 0}).values
model_df = pd.concat([dummies, df.iloc[:,[10,14]]], axis = 1)


### الجزء السادس. هندسة الميزات ووصفها 



يمكن إجراء إحدى الميزات المثيرة للاهتمام على أساس المتغيرات التي تظهر متى تم إطلاق المشروع وما هو الموعد النهائي للمشروع. أفترض أن المشاريع التي لها فترة أطول بين الإطلاق والموعد النهائي لديها فرص أكبر للنجاح. وبالتالي، دعونا ننشئ ميزة توضح عدد الأيام الفاصلة بين هذين التاريخين المهمين.


In [ ]:
df['days_diff'] = (pd.to_datetime(df["deadline"])-pd.to_datetime(df["launched"])).astype('timedelta64[D]')
df['days_diff'].head()

In [ ]:
sns.boxplot(df['days_diff'])


القيمة المتوسطة هي 29. بالنسبة لبعض المشاريع، يكون عدد الأيام بين الإطلاق والموعد النهائي 0، وهذا غريب. من باب الفضول المطلق، دعونا نتحقق مما إذا كان هناك أي مشاريع ناجحة من بين تلك المشاريع التي لم يستغرق تمويلها سوى 0 يومًا.


In [ ]:
df[(df['days_diff'] == 0) & (df['state'] == 'successful')].head()


واو! إنهم موجودون حقا!
ومن الغريب أيضًا أن تكون بعض المشاريع مدتها أكثر من 60 يومًا، حيث يمكن أن تستمر المشاريع من يوم واحد إلى 60 يومًا وفقًا للموقع الرسمي لـ Kickstarter. من المحتمل أنها مشاريع قديمة تم إطلاقها عندما اختلفت المتطلبات.
دعونا ننظر في كيفية ربط الميزة الجديدة بالمتغير المستهدف.


In [ ]:
sns.boxplot(df["state"], df['days_diff'])

In [ ]:
df.groupby('state')['days_diff'].median()

بالنسبة لكلا النوعين من المشاريع، تبلغ القيمة المتوسطة 29. ليس من المستغرب أن التوصية الرسمية من Kickstarter هي تعيين حملة لمدة 30 يومًا أو أقل، حيث أن "الحملات ذات الفترات الأقصر تحقق معدلات نجاح أعلى، وتخلق إحساسًا مفيدًا بالإلحاح حول مشروعك".
ويدعم هذا الرسم البياني، إلى حد ما، هذا الادعاء. على الرغم من وجود متوسطات متساوية للمشاريع الفاشلة والناجحة، يمكننا أن نرى أنه من بين المشاريع الفاشلة هناك حملات طويلة الأمد أكثر. ولذلك، ينبغي إضافة هذه الميزة إلى نموذجنا.
يمكن ربط ميزة أخرى قد تكون مفيدة بمتغيرات عدد الداعمين والهدف النهائي للمشروع. ماذا لو نظرنا إلى المبلغ الذي يحتاج أحد الداعمين إلى وضعه في المشروع. لنتخيل أن لدينا مشروعين. كلاهما يحتاج إلى الحصول على 15000 دولار، لكن في المشروع الأول يشارك 10 داعمين فقط، بينما في المشروع الثاني هناك 60 داعمًا. ومن الواضح أن المشروع الثاني لديه فرص أكبر للحصول على الأموال اللازمة.


In [ ]:
df["usd_by_backer"] = df["usd_goal_real"] / (df["backers"] + 1)

In [ ]:
sns.boxplot(df["state"], np.log(df['usd_by_backer'] + 1))


نعم، كما افترضت في المشاريع الفاشلة، يحتاج كل شخص إلى تقديم أموال أكثر مقارنة بالمشاريع الناجحة.
الآن، يمكننا إضافة ميزات جديدة إلى مجموعة البيانات الخاصة بنا.


In [ ]:
model_df = pd.concat([model_df, df.iloc[:,[15,16]]], axis = 1)

In [ ]:
X_train, X_test, y_train, y_test = \
    train_test_split(model_df, y, 
                     test_size=0.3, random_state=2)


وبما أن لدينا مقاييس مختلفة جدًا لمتغيراتنا، فنحن بحاجة إلى إعادة قياسها.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### الجزء 7. التحقق من الصحة وضبط المعلمات الفائقة



حسنًا، نحتاج الآن إلى تحديد المعلمات الفائقة التي سنستخدمها في نموذجنا ومن خلال تطبيق التحقق المتبادل، اختر واحدًا يعطي نتائج أفضل.
بداية، ما هو النموذج الذي سأستخدمه في المشروع؟ إن الانحدار اللوجستي يناسب مهمتنا بشكل جيد. لدينا عدد كبير من الميزات، وفي هذه الحالة ستعطي الغابات العشوائية دقة أسوأ من الانحدار اللوجستي.نظرًا لأن لدينا عددًا كبيرًا من الميزات ومسألة التجهيز الزائد خطيرة للغاية بالنسبة لنموذجنا، فنحن بحاجة إلى العثور على أفضل معلمة تشعبية C. هذه المعلمة التشعبية مسؤولة عن تنظيم النموذج. تعاقب لغة C الميزات التي لها قيم كبيرة في النموذج وتقلل من الخطأ. وبالتالي، فإنه يسمح للباحث بتقليل فرص الحصول على وضع التجهيز الزائد الذي يعمل بشكل جيد فقط مع مجموعة التدريب، بينما في حالة بيانات الاختبار فإنه يعطي نتائج سيئة إلى حد ما.
لتقييم جودة نموذجنا، سوف نستخدم درجة ROC-AUC. تتمثل فكرة منحنى ROC في تصور التنبؤات الإيجابية الحقيقية (في حالتنا، تلك المشاريع الناجحة التي تم تصنيفها بشكل صحيح) مقابل التنبؤ الإيجابي الخاطئ (تلك المشاريع التي تم تصنيفها على أنها ناجحة عن طريق الخطأ).
للتحقق من تدابيرنا، سوف نستخدم التحقق المتبادل الطبقي. في حالة التحقق المتبادل الطبقي، يتم تقسيم عينتنا إلى بعض الطيات حيث يتم توزيع جميع الفئات التي نتوقعها بالتساوي (وهذا يعني أنه في كل طية، لدينا ما يقرب من نصف المشاريع الناجحة وما يقرب من نصف المشاريع الفاشلة).


In [ ]:
lr = LogisticRegression(random_state=2, solver='liblinear')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)
Cs = np.logspace(-3, 1, 10)
reg_params = {'C': Cs}
grid_search = GridSearchCV(lr, reg_params, n_jobs=-1, scoring ='roc_auc', cv=skf)
grid_search.fit(X_train_scaled, y_train)


حسنًا، نرى أن C الذي يساوي 10 يعطي أفضل ROC-AUC. إنها حالة مثيرة للجدل بعض الشيء حيث أن 10 هي قيمة الحدود في قائمتنا، ومن المحتمل أن تعطي القيم الأعلى نتائج أفضل. 


In [ ]:
grid_search.best_params_, grid_search.best_score_


### الجزء 8. التحقق من الصحة ومنحنيات التعلم



دعونا نرسم منحنى التحقق لدينا. سأفعل ذلك بدون البحث عن الشبكة لربط العملية قليلاً.


In [ ]:
def get_auc_lr_valid(X, y, C=1.0, seed=2,ratio = 0.9):
    idx = int(round(X.shape[0] * ratio))
    lr = LogisticRegression(C=C, random_state=seed, solver='liblinear').fit(X[:idx, :], y[:idx])
    y_pred = lr.predict_proba(X[idx:, :])[:, 1]
    score = roc_auc_score(y[idx:], y_pred) 
    return score

In [ ]:
scores = []
for C in tqdm(Cs):
    scores.append(get_auc_lr_valid(X_train_scaled, y_train, C=C))

In [ ]:
score_C_1 = get_auc_lr_valid(X_train_scaled, y_train)
plt.plot(Cs, scores, 'ro-')
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('AUC-ROC')
plt.title('Regularization Parameter Tuning')
plt.axhline(y=score_C_1, linewidth=.5, color='b', linestyle='dashed') 
plt.show()


لذلك، يمكننا أن نرى هنا صورة مثيرة للاهتمام. لدينا قيمتان تعطيان الحد الأقصى لـ ROC AUCs على المحور X: 1 و10. ربما نحتاج إلى أخذ C=1 لأن C=10 يمكن أن يؤدي إلى الإفراط في تجهيز النموذج. دعونا نتحقق من ذلك في عينة الاختبار لدينا.



### الجزء التاسع. التنبؤ بالعينات المحظورة والاختبارية 


الآن، يمكننا أن نرى كيف يتنبأ نموذجنا بمجموعة الاختبار. نظرًا لأن الوضع مع قياس C لم يكن واضحًا بعد رسم درجات ROC-AUC، فسوف نقوم بتنبؤين: الأول بـ C = 10 والثاني بـ C = 1.


In [ ]:
lr = LogisticRegression(C=10, random_state=2, solver='liblinear').fit(X_train_scaled, y_train)

In [ ]:
y_pred10 = lr.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, y_pred) 

In [ ]:
lr = LogisticRegression(C=1, random_state=2, solver='liblinear').fit(X_train_scaled, y_train)

In [ ]:
y_pred1 = lr.predict_proba(X_test_scaled)[:, 1]
roc_auc_score(y_test, y_pred) 


حسنًا، لدينا نتائج متقاربة جدًا ولكن C=10 تعطي دقة أفضل قليلًا.



### الجزء العاشر. تقييم النموذج مع وصف المقاييس



دعونا نلقي نظرة على مقياسين رئيسيين للتصنيف الثنائي: الدقة والتذكر.
- يُظهر الاستدعاء حصة الإيجابيات الحقيقية من بين جميع الحالات التي يجب تصنيفها على أنها إيجابية حقيقية (لكن بعضها تم تصنيفها على أنها سلبية).
- الدقة توضح أي حصة من الإيجابيات الحقيقية مقابل جميع الحالات التي تم تصنيفها على أنها إيجابية (ولكن بعضها سلبيات في الواقع). 
في حالتنا، ستعرض Precision عدد المشاريع المصنفة على أنها ناجحة والتي تعتبر ناجحة حقًا وعدد المشاريع الفاشلة منها.  سيُظهر الاستدعاء مدى نجاح نموذجنا في توقع المشاريع الناجحة وعدد المشاريع التي تم تصنيفها بشكل خاطئ على أنها مشاريع فاشلة. 
في النتيجة التالية يمكنك رؤية التدابير المذكورة أعلاه تحسب للمشاريع الناجحة.


In [ ]:
y_pred = lr.predict(X_test_scaled)
print("Precision:{}".format(precision_recall_fscore_support(y_test, y_pred)[0][1]))
print("Recall:{}".format(precision_recall_fscore_support(y_test, y_pred)[1][1]))
print("F1:{}".format(precision_recall_fscore_support(y_test, y_pred)[2][1]))


في هذه النتيجة، يرمز الصف الأول إلى الدقة، والثاني إلى الاستدعاء، والثالث إلى درجة F1 التي تحاول إظهار مقياس متوازن بين الدقة والاستدعاء.
لذلك، نرى أن نموذجنا ليس مثاليًا ونرتكب أخطاء، ولكننا بالتأكيد نرى أن الدقة تعمل بشكل أفضل نسبيًا من الاستدعاء في نموذجنا.  هذا يعني أن نموذجنا جيد إلى حد ما في فرز المشاريع الفاشلة ونادرا ما يصنفها على أنها ناجحة، لكنه في الوقت نفسه يفقد الكثير من المشاريع الناجحة (~ 24%) ويصنفها على أنها مشاريع فاشلة. 



### الجزء 11. الاستنتاجات


في هذا المشروع، قمت بتطبيق الانحدار اللوجستي لتصنيف المشاريع الناجحة والفاشلة على Kickstarter. يمكن أن يكون هذا النموذج أداة مفيدة جدًا لممولي التمويل الجماعي المحتملين لأنه يسمح لهم باختيار فئة أفضل لمشروعهم (على سبيل المثال، من الأفضل جمع الأموال لفيلم قصير) ويمكن استخدامه لتحديد الأهداف المالية المثالية.
وفي الوقت نفسه، حتى الآن، النموذج ليس مثاليا. إنه جيد إلى حد ما في تحديد المشاريع الفاشلة وفرزها ولكنه يفقد مجموعة كبيرة من المشاريع الناجحة ويصنفها على أنها مشاريع فاشلة. بالتأكيد، يجب إصلاح هذه المشكلة: نحتاج إلى دراسة خصائص المشاريع الناجحة بشكل أفضل، والعمل على ميزات جديدة لها، وربما النظر في خوارزميات التصنيف الأخرى. أيضًا، يمكن أن تكون هذه المشكلة مرتبطة بطريقة أو بأخرى بعدم التوازن في المتغير المستهدف. لذا، فإن الطريقة الجيدة لتحسين التدابير النهائية هي موازنة حالاتنا.